# 6 · Relation extraction with Gemini  (per article)

Free-form open relation extraction over an article's **already-typed, deduplicated entities** from
nb4: every stated relation becomes a directed `[subject, relation_phrase, object]` triple with a
verbatim evidence span, where the model invents the `relation_phrase` exactly as nb2 invents type
labels. Only entities from the given list may be arguments — that frozen argument set is the whole
point of the staged decomposition, and what the e2e baseline (6c) does *not* do.

Canonicalizing the phrases to schema.org properties is nb7's job.

- **Config:** one line — `EXPERIMENT`; everything else comes from `configs/<EXPERIMENT>.json`.
- **Input:** `<id>/two_staged/1_entities/entities_typed<_isfx>[_consolidated].json` (nb4) + the raw
  article text, for the reference date and the evidence spans.
- **Output:** `<id>/two_staged/2_binary_relations/oie_relations<_osfx>.json`.

In [ ]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [ ]:
# ---- WHICH EXPERIMENT ---------------------------------------------------------------------
# ONE name decides the whole run. Every knob below comes from configs/<EXPERIMENT>.json — the SAME
# file every other notebook in both tracks reads, so a chain cannot drift apart stage to stage.
#   list what exists :  python -m experiment_config list          (run from notebooks/)
#   see one resolved :  python -m experiment_config show main
# $PHASE2_EXPERIMENT wins when it is set, so a headless batch of ablations (run_e2e.sh) can sweep
# experiments without editing a single notebook.
import os
EXPERIMENT = os.environ.get("PHASE2_EXPERIMENT") or "main"

import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "experiment_config.py").exists():
        sys.path.insert(0, str(_p)); break
import experiment_config as xcfg; importlib.reload(xcfg)

# Sets RESULTS_MODEL, BACKEND, GEMINI_MODEL, RUN_TAG, INPUT_TAG, SELECTED_ARTICLES and this
# section's own knobs: WORKERS, THINKING_BUDGET, MAX_OUTPUT_TOKENS.
CFG = xcfg.load(EXPERIMENT, globals(), "6")
# A one-off try goes RIGHT HERE, below the load, where it wins over the file.

# --- derived paths -------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR = ROOT / "results" / RESULTS_MODEL
DATASET_DIR = ROOT / "dataset/articles"   # reads <id>.txt for the REFERENCE_DATE and relation evidence

# _isfx says which nb4 run's typed entities to relate; _osfx tags what this notebook writes, and nb7
# reads that tag back as its own INPUT_TAG. No VARIANT here: VARIANT names a reification prompt arm,
# and the binary track has none.
_isfx = f"_{INPUT_TAG}" if INPUT_TAG else ""
_osfx = f"_{RUN_TAG}" if RUN_TAG else ""
OUTPUT_NAME = f"two_staged/2_binary_relations/oie_relations{_osfx}.json"

# Which per-article entity file to use as the relation arguments. The consolidated file (nb4's global
# re-consolidation) is preferred when present, because that is the vocabulary the later stages match
# against; the raw incremental one is the fallback. Both carry the nb4 run's tag, so this stage can
# only ever read the entities of the run named by INPUT_TAG — mixing two nb4 runs' entities into one
# relation set is the failure this indirection exists to prevent.
ENTITY_FILES = [f"two_staged/1_entities/entities_typed{_isfx}_consolidated.json",
                f"two_staged/1_entities/entities_typed{_isfx}.json"]

def entity_path(aid):
    for name in ENTITY_FILES:
        p = RESULTS_DIR / aid / name
        if p.exists():
            return p
    return None

_available = sorted({p.relative_to(RESULTS_DIR).parts[0]
                     for p in RESULTS_DIR.glob(f"*/two_staged/1_entities/entities_typed{_isfx}*.json")},
                    key=xcfg.article_key)
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no nb4 entity file for this INPUT_TAG):", missing)
    ARTICLE_IDS = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_IDS, (
    f"No runnable articles. Looked for */two_staged/1_entities/entities_typed{_isfx}*.json — nb4's "
    f"output for INPUT_TAG={INPUT_TAG!r} under experiment {EXPERIMENT!r}. Run nb2 -> nb4 first.")
ARTICLE_ID = ARTICLE_IDS[0]                      # the single article the demo cell below uses

print(f"\nreads nb4 run {INPUT_TAG!r} | writes run {RUN_TAG!r} -> results/{RESULTS_MODEL}/<id>/{OUTPUT_NAME}")
print(f"{len(ARTICLE_IDS)} article(s): {ARTICLE_IDS}")

In [ ]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None
LAST_USAGE = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return json.loads(resp.text)


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


## 1. Load the article's entities (relation arguments)

Every entity node from nb4 becomes one **entity card** — a stable id, its canonical surface text,
its chosen type, and its aliases — that the model may reference as a subject or object. The id map is
persisted alongside the relations so nb7 can resolve `subject_id` / `object_id` back to entity nodes.

In [ ]:
def load_entities(aid):
    """Read the article's final entity nodes (nb4) and build the id-stamped entity cards.
    Returns (entities_file_basename, cards) where each card is {id, text, type, aliases}."""
    p = entity_path(aid)
    assert p is not None, f"no entity file for {aid} (expected one of {ENTITY_FILES} under results/{aid}/)"
    data = json.loads(p.read_text())
    cards = []
    for i, e in enumerate(data.get("entities", []), 1):
        cards.append({
            "id":      f"E{i}",
            "text":    e.get("canonical_text"),
            "type":    e.get("chosen_type"),
            "aliases": e.get("aliases", []),
        })
    return p.name, cards

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline from the article header -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"

# quick look at what the demo article's arguments are
_fname, _cards = load_entities(ARTICLE_ID)
print(f"{ARTICLE_ID}: {len(_cards)} entities from {_fname}")
for c in _cards[:8]:
    print(f"  {c['id']:<4} {c['text']!r:<28} :: {c['type']}")

## 2. Open relation extraction

In [ ]:
# Free-form relation extraction over the given entities -> results/<id>/<OUTPUT_NAME>
REL_OIE_SYSTEM_PROMPT = """You are an expert Open Relation Extraction system.
You are given a TEXT to analyze, a REFERENCE_DATE (the article's publication date, ISO-8601) for context, and a list of ENTITIES already extracted from that TEXT. Each entity has an id, its surface text, its type, and optional aliases.
TASK: Extract every relation that the TEXT explicitly states between two of the given ENTITIES. Represent each relation as a directed triple [subject, relation_phrase, object], where subject and object are two different entities from the ENTITIES list. For the relation_phrase, choose freely the label that best captures the relation — you are not given a fixed set of relations: invent whatever concise phrase fits. After extraction, produce a definition for every unique relation_phrase you used.
RULES:
1. Subject and object MUST both come from the given ENTITIES list — reference them by their exact id (e.g. "E7"). NEVER introduce an entity that is not in the list, and NEVER relate an entity to itself. If two entities share the same surface text but different types, choose the id whose type fits the relation's context.
2. Direction matters: the triple reads subject --relation_phrase--> object. Orient every triple so it reads correctly (e.g. for "Acme was founded by Jane", the triple is [Jane, founded, Acme], not the reverse). Also copy the subject's and object's surface text into "subject_text" / "object_text" (exactly as they appear in the ENTITIES list) so the direction is human-checkable.
3. Only extract a relation that is explicitly stated or unambiguously entailed by the TEXT. Do NOT use outside knowledge, and do NOT guess relations that the TEXT does not support. If nothing connects two entities in the text, emit nothing for that pair.
4. Choose a concise relation_phrase — prefer a short verb or verb phrase in a canonical present-tense form (e.g. "founded", "born in", "works for", "located in", "acquired", "member of"). Keep it GENERAL and reusable: name the relation itself and do NOT pack the object/subject or qualifiers into the phrase (name the relation itself, not the relation plus its object; e.g. "located in", not "located in <place>"). Keep phrasing consistent: if two relations are the same kind of relation, give them the exact same relation_phrase (same spelling, same casing).
5. Choose the most specific relation_phrase the evidence supports, but general enough to be reusable across documents (a stable, commonly-recognized relation), not a one-off phrase that fits only this sentence. Stay grounded in the text: do not over-generalize, and do not assert a relation the evidence does not support.
6. "evidence" is the shortest verbatim span from the TEXT that states the relation — copied exactly, character-for-character, from the TEXT. "confidence" is your certainty in the triple (subject, relation_phrase, object, and direction), a value between 0 and 1; lower it when the relation is implicit, ambiguous, or the direction is uncertain.
7. The same pair of entities may hold more than one relation (emit one triple per distinct relation). The same relation_phrase may connect many different pairs. Do not emit the exact same [subject_id, relation_phrase, object_id] triple more than once.
8. After extracting all relations, collect every unique relation_phrase you used. For each, provide: (a) "definition": a concise one-sentence gloss describing the relation as a neutral association between the two kinds of entity it connects. Do NOT fix which entity is the subject and which is the object, do NOT use the words "subject"/"object", and do NOT phrase the direction — the actual direction is already carried by each extracted triple's argument order, so the definition should read the same for the relation and its inverse (e.g. "An employment relation between a person and an organization.", "A sibling relation between two people.", "A birthplace relation between a person and a place."); and (b) "synonyms": a short list of 2-5 alternative phrasings or near-synonyms of the relation. Both must be general and independent of the specific entities in the text.
9. The set of distinct relation_phrase values used in "relations" must be exactly the set defined in "relation_types": every phrase used has exactly one definition, and every defined phrase is used by at least one relation.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.
OUTPUT SCHEMA:
{
  "relations": [
    { "subject_id": "E1", "subject_text": "subject surface text", "relation_phrase": "relation label", "object_id": "E2", "object_text": "object surface text", "evidence": "verbatim span from TEXT", "confidence": <0-1> }
  ],
  "relation_types": [
    { "relation_phrase": "relation label", "definition": "A concise neutral one-sentence gloss of the relation between the two entity kinds.", "synonyms": ["alternative phrasing", "near-synonym"] }
  ]
}"""

def _entities_block(cards):
    return "\n".join(
        f'  {c["id"]} | text: "{c["text"]}" | type: {c["type"]}'
        + (f' | aka: {", ".join(c["aliases"])}' if c.get("aliases") else "")
        for c in cards)

def extract_relations(aid, write=True, verbose=True):
    """Run Gemini open relation extraction over the article's entities -> results/<id>/<OUTPUT_NAME>."""
    text = (DATASET_DIR / f"{aid}.txt").read_text()
    ref_date = _reference_date(text)
    fname, cards = load_entities(aid)
    ids = {c["id"] for c in cards}
    by_id = {c["id"]: c for c in cards}
    if verbose:
        print(f"[{aid}] ~{len(text)//4} tokens; {len(cards)} entities; REFERENCE_DATE={ref_date}; calling {GEMINI_MODEL} ...")
    user_message = (
        (f"REFERENCE_DATE: {ref_date}\n\n" if ref_date else "")
        + f"ENTITIES:\n{_entities_block(cards)}\n\nTEXT:\n{text}")
    oie = gemini_json(REL_OIE_SYSTEM_PROMPT, user_message)

    # Keep only relations whose subject and object are both real entity ids and are distinct.
    clean, dropped, seen = [], 0, set()
    for r in oie.get("relations", []):
        s, o = r.get("subject_id"), r.get("object_id")
        key = (s, r.get("relation_phrase"), o)
        if s in ids and o in ids and s != o and key not in seen:
            seen.add(key)
            r["subject_text"] = by_id[s]["text"]        # trust the id, re-stamp the surface text
            r["object_text"]  = by_id[o]["text"]
            clean.append(r)
        else:
            dropped += 1
    # Keep only relation_types actually used (and warn about any used phrase left undefined).
    used = list(dict.fromkeys(r["relation_phrase"] for r in clean))
    rtypes = {t["relation_phrase"]: t for t in oie.get("relation_types", []) if t.get("relation_phrase") in used}
    for ph in used:
        rtypes.setdefault(ph, {"relation_phrase": ph, "definition": "", "synonyms": []})

    out = {"article_id": aid, "entities_file": fname,
           "entities": cards, "relations": clean,
           "relation_types": [rtypes[ph] for ph in used]}
    if write:
        out_dir = RESULTS_DIR / aid
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / OUTPUT_NAME).write_text(json.dumps(out, indent=2, ensure_ascii=False))
    if verbose:
        note = f" ({dropped} dropped: bad/duplicate ids)" if dropped else ""
        print(f"  latency {last_latency_s:.2f}s | {len(clean)} relations{note}, "
              f"{len(used)} distinct relation phrases -> results/{aid}/{OUTPUT_NAME}")
    return out

# oie = extract_relations(ARTICLE_ID)

## 3. Inspect the demo article

In [ ]:
# Extract relations for the single demo article and print them as readable triples.
oie = extract_relations(ARTICLE_ID)
print("-" * 74)
for r in oie["relations"]:
    print(f'  {r["subject_text"]!r}  --{r["relation_phrase"]}-->  {r["object_text"]!r}   (conf {r.get("confidence")})')
    print(f'      ev: "{r.get("evidence","")}"')
print("-" * 74)
print("relation phrases:")
for t in oie["relation_types"]:
    print(f'  {t["relation_phrase"]:<22} {t.get("definition","")}')

## 4. Batch — process the selected articles (`ARTICLE_IDS`)

In [ ]:
# Runs relation extraction for each article in ARTICLE_IDS (choose them via SELECTED_ARTICLES in the
# config cell). Re-runs overwrite existing outputs. (Property canonicalization comes next, in nb7.)
import time
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_t0 = time.perf_counter()

for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    with nb_parallel.usage_scope(aid):      # tag this article's calls so the report breaks down per article
        extract_relations(aid)
print(f"\ndone — extracted relations for {len(ARTICLE_IDS)} article(s)")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
nb_parallel.usage_report(f"nb6 · relation extraction · {EXPERIMENT}", wall_seconds=time.perf_counter() - _t0,
                         write=RESULTS_DIR / f"_global/token_usage_nb6_oie_relations{_osfx}.json")

## Run metrics — the numbers this stage produced

`run_metrics.oie` re-reads what nb2 and nb6 wrote: entity and type counts, temporal-normalization
coverage, relation and phrase counts, relations per entity, and the token cost. Saved to
`_global/metrics_oie<_osfx>.json` so the numbers survive the kernel. (nb2 and nb6 share one metrics
file, because the second stage is meaningless without the first — the relation counts are per
*this* entity set.)

In [ ]:
# Collect + persist the staged track's open-extraction numbers. Pure reading — no LLM call. Reads
# nb6's relations too when they exist, so re-running this cell after nb6 fills the second half in.
import importlib, run_metrics
importlib.reload(run_metrics)

_m = run_metrics.oie(RESULTS_DIR, ARTICLE_IDS, osfx=_osfx, experiment=EXPERIMENT)
run_metrics.show(_m)

# save() also CLOSES this stage in the run manifest — the record of WHICH run these numbers
# came from: the resolved config, the git commit, the article order, every output file and
# the token spend.  results/<model>/_runs/<run_id>.json  ·  python -m run_manifest list
print("\nsaved ->", run_metrics.save(_m, RESULTS_DIR, _osfx, "binary",
                                     section="6", articles=ARTICLE_IDS))